## **Bagging**

#### **Bagging Regressor**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv('china_used_cars.csv')
drop_cols = ['price', 'mileage_km', 'log_mileage', 'mileage_per_year', 'year', 'month']
X = df.drop(columns=drop_cols)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline: a single, unconstrained decision tree
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)
p_tree = tree.predict(X_test)
print("Single tree -> MAE:", mean_absolute_error(y_test, p_tree), " R2:", r2_score(y_test, p_tree))

# Bagging: 200 trees, each on an 80% bootstrap sample
bag = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=200,
    max_samples=0.8,
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
bag.fit(X_train, y_train)
p_bag = bag.predict(X_test)
print("Bagging  -> MAE:", mean_absolute_error(y_test, p_bag), " R2:", r2_score(y_test, p_bag))
print("OOB R2:", bag.oob_score_)

Single tree -> MAE: 12486.153016597103  R2: 0.5947655466692935
Bagging  -> MAE: 12700.15000655024  R2: 0.6818649938846035
OOB R2: 0.29803456004428397


In [2]:
for n in [1, 5, 10, 50, 100, 200]:
    b = BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),
                          n_estimators=n, random_state=42, n_jobs=-1)
    b.fit(X_train, y_train)
    p = b.predict(X_test)
    print(f"n_estimators={n:3d} -> MAE: {mean_absolute_error(y_test,p):.1f}  R2: {r2_score(y_test,p):.4f}")

n_estimators=  1 -> MAE: 19350.1  R2: -1.6213
n_estimators=  5 -> MAE: 11594.2  R2: 0.6457
n_estimators= 10 -> MAE: 14040.4  R2: 0.5371
n_estimators= 50 -> MAE: 12414.9  R2: 0.6862
n_estimators=100 -> MAE: 12152.7  R2: 0.7157
n_estimators=200 -> MAE: 12027.3  R2: 0.7401


In [3]:
from sklearn.linear_model import LinearRegression

bag_lr = BaggingRegressor(estimator=LinearRegression(), n_estimators=100, random_state=42, n_jobs=-1)
bag_lr.fit(X_train, y_train)
p_lr = bag_lr.predict(X_test)
print("Bagging LR -> MAE:", mean_absolute_error(y_test, p_lr), " R2:", r2_score(y_test, p_lr))

lr = LinearRegression().fit(X_train, y_train)
p0 = lr.predict(X_test)
print("Single LR  -> MAE:", mean_absolute_error(y_test, p0), " R2:", r2_score(y_test, p0))

Bagging LR -> MAE: 34633.574004463175  R2: 0.22512553856756545
Single LR  -> MAE: 34686.279347295764  R2: 0.2235239404569237


#### **Bagging Classifier**

In [4]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

drop_cols = ['is_electric', 'battery_capacity_kwh', 'motor_power_kw',
             'price', 'mileage_km', 'log_mileage', 'mileage_per_year', 'year', 'month']
X = df.drop(columns=drop_cols)
y = df['is_electric']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print("Single tree acc:", accuracy_score(y_test, tree.predict(X_test)))

bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=200, bootstrap=True, oob_score=True, random_state=42, n_jobs=-1
)
bag_clf.fit(X_train, y_train)
print("Bagging acc:", accuracy_score(y_test, bag_clf.predict(X_test)), " OOB:", bag_clf.oob_score_)

Single tree acc: 0.989247311827957
Bagging acc: 0.989247311827957  OOB: 0.99043919928294
